In [ ]:
#| default_exp datasources.base
%load_ext autoreload
%autoreload 2

import sys,os
from pathlib import Path

In [ ]:
# Insert in Path Project Directory
sys.path.insert(0, str(Path().cwd().parent))
os.chdir(Path.cwd().parent / "extracao")

# Classe Base
> Módulo para encapsular a extração e processamento comum às diferentes fontes de dados

In [ ]:
# | export
import json
import re
from dataclasses import dataclass
from functools import cached_property, partial
from typing import Tuple, Union, List, Any

import pandas as pd
from rich import print as pp
from dotenv import find_dotenv, load_dotenv
from fastcore.xtras import Path, listify
from pyarrow import ArrowInvalid, ArrowTypeError
from tqdm.auto import tqdm

In [ ]:
# | export
load_dotenv(find_dotenv(), override=True)
tqdm.pandas()
pd.options.mode.copy_on_write = True

True

In [ ]:
# | hide: true
# | eval:false
__file__ = Path.cwd().parent / "extracao" / "datasources.py"

In [ ]:
# | export
@dataclass
@dataclass
class Base:
	folder: Union[str, Path] = Path(__file__).parent / 'arquivos' / 'saida'
	read_cache: bool = False

	def _read(self, stem: str, backend: str = 'pyarrow') -> pd.DataFrame:
		"""Lê o dataframe formado por self.folder / self.stem.parquet.gzip"""
		file = Path(f'{self.folder}/{stem}.parquet.gzip')
		try:
			return pd.read_parquet(file, dtype_backend=backend)
		except (ArrowInvalid, FileNotFoundError) as e:
			raise ValueError(f'Error when reading {file}') from e

	def _save(self, df: pd.DataFrame, folder: Union[str, Path], stem: str) -> pd.DataFrame:
		"""Format, Save and return a dataframe"""
		folder = Path(folder)
		folder.mkdir(parents=True, exist_ok=True)
		try:
			file = Path(f'{folder}/{stem}.parquet.gzip')
			df.astype('category').to_parquet(
				file, compression='gzip', index=False, engine='pyarrow'
			)
		except (ArrowInvalid, ArrowTypeError) as e:
			raise Exception(f'Não foi possível salvar o arquivo parquet') from e
		return df

	@cached_property
	def df(self) -> pd.DataFrame:
		try:
			return self._read(self.stem)
		except (ArrowInvalid, FileNotFoundError) as e:
			raise ValueError(f'Não foi possível ler o arquivo parquet {self.stem}') from e

	@staticmethod
	def parse_bw(
		bw: str,  # Designação de Emissão (Largura + Classe) codificada como string
	) -> Tuple[Union[str, Any], Union[str, Any]]:  # Largura e Classe de Emissão
		"""Parse the bandwidth string"""
		if match := re.match(RE_BW, bw):
			multiplier = BW[match[2]]
			if mantissa := match[3]:
				number = float(f'{match[1]}.{mantissa}')
			else:
				number = float(match[1])
			classe = str(match[4])
			if not classe:
				classe = pd.NA
			return str(multiplier * number), classe
		return pd.NA, pd.NA

	# @cached_property
	# def discarded(self) -> pd.DataFrame:
	# 	return pd.DataFrame(columns=self.columns)

	# def append2discarded(self, df: pd.DataFrame) -> None:
	# 	"""Receives one of more dataframes and append to the discarded dataframe"""
	# 	if not self.discarded.empty:
	# 		self.discarded = pd.concat([self.discarded, df], ignore_index=True, copy=False)
	# 	else:
	# 		self.discarded = df

	@staticmethod
	def register_log(
		df: pd.DataFrame,
		processing: str,
		column: Union[str, None] = None,
		row_filter: Union[pd.Series, None] = None,
	):
		"""Register a log in the dataframe"""
		if row_filter is None:
			row_filter = pd.Series(True, index=df.index)
		elif not row_filter.any():
			return
		if 'Log' not in df:
			df['Log'] = '[]'
		else:
			df['Log'] = df['Log'].astype('string', copy=False).fillna('[]')
			df['Log'] = df['Log'].str.replace('^$', r'[]', regex=True)
		pp(f'[bold green]Logging:[/bold green] [italic]{processing}')
		log_function = partial(Base.format_log, processing=processing, column=column)
		if column is not None:
			df_logged = df.loc[row_filter, ['Log', column]].copy()
			df_logged[column] = df_logged[column].astype('string', copy=False).fillna('')
		else:
			df_logged = df.loc[row_filter, ['Log']]
		df.loc[row_filter, 'Log'] = df_logged.progress_apply(log_function, axis=1)

	@staticmethod
	def format_log(
		row: pd.Series,
		processing: str,
		column: Union[str, None] = None,
	) -> str:
		"""Translate log string into dict, update it and reformats it a log message
		It's assumed the typing in the signature is correct
		"""
		log = listify(eval(row.loc['Log']))
		new_log = {'Processamento': processing}
		if column is not None:
			new_log.update({'Coluna': column, 'Original': row.loc[column]})
		log.append(new_log)
		return json.dumps(log, ensure_ascii=False)

	@property
	def columns(self):
		raise NotImplementedError("Subclasses devem implementar a propriedade 'columns'")

	@property
	def cols_mapping(self):
		raise NotImplementedError("Subclasses devem implementar a propriedade 'cols_mapping'")

	@property
	def stem(self):
		raise NotImplementedError('Subclasses devem setar a propriedade stem')

	def extraction(self) -> pd.DataFrame:
		raise NotImplementedError('Subclasses devem implementar o método extract')

	def _format(
		self,
		df: pd.DataFrame,  # DataFrame com os dados de Estações
	) -> pd.DataFrame:  # DataFrame formatado
		"""Formata, limpa e padroniza os dados provenientes da query no banco"""
		raise NotImplementedError('Subclasses devem implementar o método _format')

	def update(self):
		df = self.extraction()
		if not self.read_cache:
			self._save(df.drop('Log', axis=1), self.folder, f'{self.stem}_raw')
		self.df = self._format(df)

	def save(self, folder: Union[str, Path, None] = None):
		if folder is None:
			folder = self.folder
		self._save(self.df, folder, self.stem)
		# self._save(self.discarded, folder, f'{self.stem}_discarded')